In [ ]:
import pandas as pd

# Load df_combined.csv into a DataFrame called df
df = pd.read_csv("df_combined.csv")

In [ ]:
import numpy as np

# Ensure there is a 'data_source' column; adjust if named differently
source_col = "data_source"
if source_col not in df.columns:
    raise ValueError(f"Column '{source_col}' not found in dataframe.")

# Filter sources with at least 5 articles
source_counts = df[source_col].value_counts()
eligible_sources = source_counts[source_counts >= 5].index

# Subset dataframe to only eligible sources
eligible_df = df[df[source_col].isin(eligible_sources)].copy()

# Sample at least 5 articles per source first
frames = []
for source in eligible_sources:
    source_df = eligible_df[eligible_df[source_col] == source]
    sample_size = min(5, len(source_df))
    sampled = source_df.sample(n=sample_size, random_state=42)
    frames.append(sampled)

base_sample = pd.concat(frames, ignore_index=True)
remaining_needed = max(0, 100 - len(base_sample))

# For the rest, sample randomly from eligible_df excluding already selected rows
if remaining_needed > 0:
    remaining_df = eligible_df.drop(base_sample.index)
    additional_sample = remaining_df.sample(
        n=min(remaining_needed, len(remaining_df)), random_state=42, replace=False
    )
    trial_df = pd.concat([base_sample, additional_sample], ignore_index=True)
else:
    trial_df = base_sample

# If trial_df is still larger than 100 (due to many eligible sources), subsample to 100
if len(trial_df) > 100:
    trial_df = trial_df.sample(n=100, random_state=42, replace=False).reset_index(drop=True)
else:
    trial_df = trial_df.reset_index(drop=True)

print(f"trial_df shape: {trial_df.shape}")

In [ ]:
import json
import pandas as pd

# Work on a copy; do not modify the original df
work = df_trial.copy()

# Ensure we have string columns; fill NaN with "" and strip whitespace
for col in ["title", "Date", "Text"]:
    if col in work.columns:
        work[col] = work[col].fillna("").astype(str).str.strip()
    else:
        work[col] = ""

n_original = len(work)

# Drop rows where Text is empty after cleaning
work = work[work["Text"].str.len() > 0].copy()
n_kept = len(work)
n_dropped = n_original - n_kept
print(f"Dropped {n_dropped} rows with empty Text.")

# Sequential custom_id: a_000001, a_000002, ...
work["custom_id"] = [f"a_{i:06d}" for i in range(1, n_kept + 1)]

# Build the "input" field: Title: {title}\nDate: {Date}\n\nText: {Text}
work["input_text"] = (
    "Title: " + work["title"] + "\nDate: " + work["Date"].astype(str) + "\n\nText: " + work["Text"]
)

# Build the full JSON object per row for the batch API
batch_lines = []
for _, row in work.iterrows():
    obj = {
        "custom_id": row["custom_id"],
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": "gpt-5-mini",
            "instructions": "PLACEHOLDER_PROMPT",
            "input": row["input_text"],
            "temperature": 0,
        },
    }
    batch_lines.append(json.dumps(obj, ensure_ascii=False))

# Write JSONL (UTF-8)
with open("batch_input.jsonl", "w", encoding="utf-8") as f:
    for line in batch_lines:
        f.write(line + "\n")

# Mapping file: custom_id, title, Date, URL, datasource
mapping_cols = ["custom_id", "title", "Date", "URL", "datasource"]
mapping_df = work[[c for c in mapping_cols if c in work.columns]].copy()
if "URL" not in work.columns:
    mapping_df["URL"] = ""
if "datasource" not in work.columns:
    mapping_df["datasource"] = ""
mapping_df = mapping_df[["custom_id", "title", "Date", "URL", "datasource"]]
mapping_df.to_csv("batch_mapping.csv", index=False, encoding="utf-8")

# Summary and preview
print(f"Original rows: {n_original}")
print(f"Rows kept:     {n_kept}")
print(f"Rows dropped:  {n_dropped}")
print("\nFirst 2 JSONL lines (preview):")
for line in batch_lines[:2]:
    print(line)